### 本地 IDE 运行与环境配置（本地运行必读，若在服务器运行可跳过）

若在本地运行本项目，请先完成以下步骤：

- 环境准备（任选其一）
  - conda：`conda create -n camels python=3.10 -y && conda activate camels && pip install -r requirements.txt`
  - venv（Windows PowerShell）：`python -m venv .venv`，然后 `.\\.venv\\Scripts\\Activate.ps1`，再 `pip install -r requirements.txt`
  - venv（macOS/Linux）：`python3 -m venv .venv`，然后 `source .venv/bin/activate`，再 `pip install -r requirements.txt`

- 配置 hydrodataset 的数据根目录（本地必做）
  - Windows：在 `%USERPROFILE%\\.hydrodataset\\settings.txt` 写入 CAMELS 数据根目录的绝对路径（仅一行，不加引号），如 `D:\\data\\hydrodataset`
  - macOS/Linux：在 `~/.hydrodataset/settings.txt` 写入绝对路径，如 `/data/hydrodataset`
  - 完成后在下方单元格执行：

```python
import hydrodataset
print("数据根目录:", hydrodataset.ROOT_DIR)
```

- 数据与包下载地址
  - CAMELS 数据下载地址：<https://zenodo.org/records/15529996>
  - hydrodataset 包：<https://github.com/iHeadWater/hydrodataset>

- 目录结构应包含
  - `ROOT_DIR/camels/camels_us/camels_streamflow.nc`
  - `ROOT_DIR/camels/camels_us/camels_daymet_forcing.nc`
  - `ROOT_DIR/camels/camels_us/camels_attributes_v2.0.feather`

- 常见问题
  - `netCDF4` 安装失败：优先使用 `requirements.txt`；仍失败可尝试 `conda install -c conda-forge netcdf4`
  - Windows 路径规范：避免中文与空格；`settings.txt` 仅填写绝对路径
  - 找不到数据：确认 `settings.txt` 指向目录下存在 `camels/camels_us/*.nc` 与属性 `*.feather` 文件


### 获取CAMELS数据

在开始正式实现 LSTM-CAMELS 之前，我们需要先获取 CAMELS 数据。本节说明在服务器与本地两种环境下如何获取并读取 CAMELS 数据。

#### 方式一：服务器（推荐）
- 为了方便大家使用 CAMELS 数据，我们已经将数据下载到平台服务器，并打包了读取 CAMELS 数据的代码，且内置在当前的 Python 环境中，所以可以直接通过简单的调用来读取CAMELS 数据。平台已预置数据与依赖，可直接运行，不需要手动下载与配置。
- 建议先在上方单元检查 `hydrodataset.ROOT_DIR` 是否为有效目录。

#### 方式二：本地 IDE
- 按顶部“本地 IDE 运行与环境配置”完成环境与 `settings.txt` 配置。
- 按 README 的下载链接下载并解压 CAMELS 数据到 `ROOT_DIR`，目录应包含：
  - `ROOT_DIR/camels/camels_us/camels_streamflow.nc`
  - `ROOT_DIR/camels/camels_us/camels_daymet_forcing.nc`
  - `ROOT_DIR/camels/camels_us/camels_attributes_v2.0.feather`
- 可先运行“本地数据文件校验”单元确认文件齐全。

完成上述任一方式后，即可继续使用 `xarray` 懒加载读取数据。


### NetCDF 格式简介

我们会经常遇见 `.nc` 文件，也就是 NetCDF 格式文件。

从数学上来说，NetCDF 存储的数据就是**多个多自变量的单值函数**。一个函数用公式来说就是 `f(x, y, z, …) = value`。

- 函数的自变量 x, y, z 等在 NetCDF 中叫做**维 (dimension)** 或**坐标轴 (axis)**
- 函数值 value 在 NetCDF 中叫做**变量 (Variables)**

一个 NetCDF 文件的结构包括以下对象：
- **变量 (Variables)**：变量对应着真实的物理数据
- **维 (dimension)**：一个维对应着函数中的某个自变量，或者说函数图象中的一个坐标轴，典型地是三维（经纬度+时间）
- **属性 (Attribute)**：属性是对变量值和维的具体物理含义的注释

NetCDF 文件中的数据以数组形式存储。例如：
- 某个位置处随时间变化的温度以一维数组的形式存储
- 某个区域内在指定时间的温度以二维数组的形式存储
- 三维 (3D) 数据（如某个区域内随时间变化的温度）或四维 (4D) 数据（如某个区域内随时间和高度变化的温度）以一系列二维数组的形式存储


### 使用 xarray 读取 NetCDF 文件

读取 `.nc` 文件有很多种方法，这里演示 Python 中最常用的使用 `xarray` 读取的方式。

我们通过 `xarray` 工具包中的 `.open_dataset()` 函数来懒加载 `.nc` 文件（不一次性读入内存）。

下文示例读取 CAMELS-US 的流域径流数据文件；无论服务器或本地，只要 `hydrodataset.ROOT_DIR` 指向正确的根目录，路径拼接方式相同。


### 配置 hydrodataset 数据目录

首次使用需要配置 CAMELS 数据集的文件路径。

#### 方法1：使用服务器（如 jupyterhub）

在平台 jupyterlab 首页打开终端，然后输入以下命令：

```bash
# 进入配置文件所在文件夹
cd ~/.hydrodataset
# 使用 vim 打开配置文件
vim settings.txt
```

打开后，按 `i` 键，将 vim 编辑器调整至 INSERT 模式，然后输入 `/ftproot`（ftproot 是服务器上放置公共数据的默认文件夹）。

然后按 `:` （英文输入法下的冒号）进入命令模式，输入 `wq` 并按回车键，就能写入（即保存）并退出了。

配置完成后，需要重新打开当前文件的 kernel，重新执行下面的导入语句。

#### 方法2：本地 IDE（Windows/macOS/Linux）

- 创建或编辑设置文件：
  - Windows: `%USERPROFILE%\\.hydrodataset\\settings.txt`
  - macOS/Linux: `~/.hydrodataset/settings.txt`
- 文件仅一行：填写 CAMELS 数据根目录的绝对路径（不加引号），如：
  - Windows: `D:\\data\\hydrodataset`
  - macOS/Linux: `/data/hydrodataset`
- 完成后在下方单元执行 `import hydrodataset; print(hydrodataset.ROOT_DIR)` 验证路径是否生效。


In [ ]:
# 导入 hydrodataset 并检查数据根目录（服务器/本地均适用）
import hydrodataset
print("数据根目录:", hydrodataset.ROOT_DIR)
# 若目录为空或不存在，请返回上方章节检查 settings.txt 或平台预设路径

In [ ]:
# 本地数据文件校验（缺失将抛出清晰提示）
import os
from pathlib import Path
import hydrodataset

root_dir = Path(hydrodataset.ROOT_DIR)
required_files = [
    root_dir / "camels" / "camels_us" / "camels_streamflow.nc",
    root_dir / "camels" / "camels_us" / "camels_daymet_forcing.nc",
    root_dir / "camels" / "camels_us" / "camels_attributes_v2.0.feather",
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "以下必需数据文件未找到，请检查 settings.txt 配置与数据解压路径:\n" + "\n".join(missing)
    )
else:
    print("本地数据文件检查通过，共", len(required_files), "个文件存在。")


### 读取 CAMELS NetCDF 数据示例

现在数据已经配置好了，我们就能读取一个 nc 文件试试了。


In [ ]:
import xarray as xr

dataset_dir = hydrodataset.ROOT_DIR
camels_streamflow = xr.open_dataset(
    dataset_dir.joinpath("camels", "camels_us", "camels_streamflow.nc")
)
camels_streamflow


可以看见 NetCDF 文件的维度 (Dimensions)、坐标 (Coordinates)、属性 (Attributes) 与变量 (Data variables)。

查看有哪些流域：


In [ ]:
# 查看流域列表
camels_streamflow.basin


成功后，接下来我们就可以试试读取CAMELS数据集了

In [ ]:
# 配置与导入（如未配置会报错，已配置可直接使用）
import hydrodataset
from hydrodataset.camels import Camels
import os

# 如未配置，将出现配置文件未找到的提示；已配置则可忽略此段提示



In [ ]:
# 创建 CAMELS-US 数据对象（示例）
camels_us_path = os.path.join("camels", "camels_us")
us_region = "US"
camels_us = Camels(camels_us_path, region=us_region)
camels_us.camels_sites.head()


### 使用缓存的高效数据格式

- 径流与气象数据采用 NetCDF（nc）格式，建议用 `xarray` 懒加载，避免一次性加载到内存。
- 属性数据采用 feather 格式，可用 `pandas` 读取。

下方示例展示数据目录、缓存目录，以及如何读取流量与气象数据集。


In [ ]:
import pandas as pd
import xarray as xr

# 数据与缓存目录
data_dir = camels_us.data_source_dir
cache_dir = hydrodataset.CACHE_DIR
print("data_dir:", data_dir)
print("cache_dir:", cache_dir)

# 读取数据集
streamflow_ds = xr.open_dataset(data_dir.joinpath("camels_streamflow.nc"))
forcing_ds = xr.open_dataset(data_dir.joinpath("camels_daymet_forcing.nc"))
attrs = pd.read_feather(data_dir.joinpath("camels_attributes_v2.0.feather"))

streamflow_ds, forcing_ds, attrs.head()


In [ ]:
# 示例：选择一个流域与时间段进行可视化（如需）
_ = streamflow_ds.sel(basin="01013500", time=slice("2000-06-01", "2001-05-31")).to_pandas().plot()
